# Трекинг класса человек  

### Вывод информации на экран

In [1]:
from ultralytics import YOLO
import cv2
import numpy as np
import signal
from collections import defaultdict

import sys

In [11]:
# Загрузка добученной модели YOLOv8
model = YOLO("runs/result/best.pt")

In [4]:
# Загрузка изображения
#image = cv2.imread('2.png')
image = cv2.imread('E://СИИ/CursachProject/AI-Flow-Detecting/ai-core/dataset/val_split/event_20251203_172807_611580.jpg')

# Детекция людей (класс '0' в COCO = человек)
results = model(image, classes=[0])  

# Визуализация результатов
annotated_image = results[0].plot()  # Рисует bounding boxes
cv2.imshow('Detected People', annotated_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Подсчёт количества людей
people_count = len(results[0].boxes)
print(f"Количество людей на фото: {people_count}")



0: 384x640 5 humans, 106.7ms
Speed: 2.3ms preprocess, 106.7ms inference, 31.0ms postprocess per image at shape (1, 3, 384, 640)
Количество людей на фото: 5


In [ ]:
# URL видеопотока (убраны лишние пробелы в конце!)
# https://restreamer.vms.evo73.ru/918335436b92ac26/stream.m3u8
# https://restreamer.vms.evo73.ru/24c3036fe19a150a/stream.m3u8
stream_url = "https://restreamer.vms.evo73.ru/918335436b92ac26/stream.m3u8"

# Открываем видеопоток
cap = cv2.VideoCapture(stream_url)

if not cap.isOpened():
    print("❌ Ошибка: Не удалось открыть видеопоток. Проверь URL и соединение.")
    exit()

print("✅ Видеопоток запущен. Детекция людей через YOLOv8...")

while True:
    ret, frame = cap.read()

    if not ret:
        print("⚠️ Не удалось получить кадр. Поток может быть разорван.")
        break

    # Детекция ТОЛЬКО людей (класс 0 в COCO)
    results = model.predict(frame, classes=[0], conf=0.5, verbose=False)

    # Наносим bounding boxes и метки
    annotated_frame = results[0].plot()

    # Подсчёт обнаруженных людей
    people_count = len(results[0].boxes)
    cv2.putText(annotated_frame, f'Людей: {people_count}', (10, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2, cv2.LINE_AA)

    # Показываем кадр
    cv2.imshow('YOLOv8 — Детекция людей в реальном времени', annotated_frame)

    # Выход по клавише 'q' или пробелу
    if cv2.waitKey(1) & 0xFF in [ord('q'), ord(' ')]:
        break

# Освобождение ресурсов
cap.release()
cv2.destroyAllWindows()
print("⏹️ Работа завершена.")

✅ Видеопоток запущен. Детекция людей через YOLOv8...
⏹️ Работа завершена.


In [ ]:
import cv2
from ultralytics import YOLO
import random

def process_video_with_tracking(model, input_source, show_video=True, save_video=False, output_video_path="output_video.mp4"):
    """ Функция для обработки видео с детекцией объектов и трекингом. """

    # Открыть источник видео (видеофайл или поток)
    cap = cv2.VideoCapture(input_source)

    if not cap.isOpened():
        raise Exception("Ошибка: Не удалось открыть видео или поток.")

    # Получить параметры видео
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Настройка записи видео если нужно сохранять
    if save_video:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

    while True:
        ret, frame = cap.read()

        if not ret:
            print("⚠️ Не удалось получить кадр. Поток может быть прерван.")
            break

        # Детекция объектов
        results = model.track(
            frame,
            iou=0.4,           # Порог пересечения для трекинга
            conf=0.5,          # Порог уверенности
            persist=True,      # Сохранять треки между кадрами
            imgsz=608,         # Размер изображения для обработки
            verbose=False,     # Не выводить подробную информацию
            tracker="botsort.yaml"  # Тип трекера
        )

        # Наносим bounding boxes и метки
        annotated_frame = results[0].plot()

        # Подсчёт обнаруженных людей (класс 0)
        people_count = len(results[0].boxes)
        cv2.putText(annotated_frame, f'People: {people_count}', (10, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2, cv2.LINE_AA)

        # Если есть трекинг, добавляем ID объектов
        if results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
            ids = results[0].boxes.id.cpu().numpy().astype(int)

            for box, id in zip(boxes, ids):
                # Генерация случайного цвета для каждого объекта на основе его ID
                random.seed(int(id))
                color = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))

                # Рисование прямоугольника
                cv2.rectangle(annotated_frame, (box[0], box[1]), (box[2], box[3]), color, 2)
                # Добавление текста с ID
                cv2.putText(
                    annotated_frame,
                    f"Id {id}",
                    (box[0], box[1]),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0, 255, 255),
                    2,
                )

        # Сохранение кадра в файл если нужно
        if save_video:
            out.write(annotated_frame)

        # Показ кадра если нужно
        if show_video:
            # Уменьшение размера для удобства просмотра
            annotated_frame = cv2.resize(annotated_frame, (0, 0), fx=0.75, fy=0.75)
            cv2.imshow("YOLOv11 — Обнаружение людей в реальном времени", annotated_frame)

        # Выход по нажатию клавиши 'q'
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    # Освобождение ресурсов
    cap.release()
    if save_video:
        out.release()

    # Закрытие всех окон OpenCV
    cv2.destroyAllWindows()

# Загрузка модели YOLOv11
model = YOLO('runs/result/preprocessdetect.pt')
model.fuse()  # Оптимизация модели

# Путь к видео или URL потока
# input_source = "test.mp4"  # Локальный файл
input_source = "https://restreamer.vms.evo73.ru/918335436b92ac26/stream.m3u8"  # Видеопоток

# Запуск обработки видео
process_video_with_tracking(
    model,
    input_source,
    show_video=True,
    save_video=False,
    output_video_path="output_video.mp4"
)

YOLO11m summary (fused): 125 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs


In [4]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO
import seaborn as sns
import pandas as pd
from pathlib import Path
import time
import os

# Создаем папку ai-core, если её нет
os.makedirs('ai-core', exist_ok=True)

# ЗАГРУЗКА МОДЕЛИ
model_path = 'runs/result/2final.pt'
model = YOLO(model_path)
model.fuse()

# ВАЛИДАЦИЯ МОДЕЛИ
data_config = {
    'nc': 1,
    'names': ['human'],
    'path': 'dataset',
    'train': 'train_split', 
    'val': 'val_split'
}

# Создаем data.yaml файл
data_yaml_path = 'custom_data.yaml'
with open(data_yaml_path, 'w') as f:
    f.write(f"nc: {data_config['nc']}\n")
    f.write(f"names: {data_config['names']}\n")
    f.write(f"path: {data_config['path']}\n")
    f.write(f"train: {data_config['train']}\n")
    f.write(f"val: {data_config['val']}\n")

# Проверяем существование директорий
dataset_path = Path(data_config['path'])
train_path = dataset_path / data_config['train']
val_path = dataset_path / data_config['val']

try:
    results = model.val(
        data=data_yaml_path,
        split='val',
        imgsz=608,
        conf=0.5,
        iou=0.45,
        device='cpu',
        plots=True,      # Графики будут сохранены автоматически в runs/val/
        save_json=True,
        verbose=False
    )

    metrics = getattr(results, 'results_dict', {})
    map50 = metrics.get('metrics/mAP50(B)', 0)
    map50_95 = metrics.get('metrics/mAP50-95(B)', 0)
    precision = metrics.get('metrics/precision(B)', 0)
    recall = metrics.get('metrics/recall(B)', 0)
    f1_score = 2 * (precision * recall) / (precision + recall) if precision > 0 and recall > 0 else 0

except Exception as e:
    # Ошибка валидации — записываем в файл, не выводим в консоль
    with open('ai-core/validation_error.log', 'w') as f:
        f.write(f"Ошибка валидации: {e}")
    metrics = {}

# ВИЗУАЛИЗАЦИЯ МЕТРИК — СОХРАНЯЕМ В ai-core/
def create_metrics_plots(metrics_data, model_name):
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle(f'Анализ качества модели: {model_name}', fontsize=16, fontweight='bold')
    
    # 1. Основные метрики
    main_metrics_labels = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall']
    main_metrics_values = [
        metrics_data.get('metrics/mAP50(B)', 0),
        metrics_data.get('metrics/mAP50-95(B)', 0),
        metrics_data.get('metrics/precision(B)', 0),
        metrics_data.get('metrics/recall(B)', 0)
    ]
    
    bars = axes[0, 0].bar(main_metrics_labels, main_metrics_values, 
                         color=['#1f77b4', '#2ca02c', '#d62728', '#ff7f0e'])
    axes[0, 0].set_title('Основные метрики качества')
    axes[0, 0].set_ylabel('Значение')
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    for bar, value in zip(bars, main_metrics_values):
        height = bar.get_height()
        axes[0, 0].text(bar.get_x() + bar.get_width()/2, height + 0.02, 
                       f'{value:.3f}', ha='center', va='bottom')

    # PR-кривая
    try:
        if hasattr(results, 'box') and hasattr(results.box, 'pr_array'):
            pr_array = results.box.pr_array
            if pr_array is not None and len(pr_array) > 0:
                axes[0, 1].plot(pr_array[:, 1], pr_array[:, 0], 'b-', linewidth=2)
                axes[0, 1].fill_between(pr_array[:, 1], pr_array[:, 0], alpha=0.2)
                axes[0, 1].set_title('Precision-Recall Curve')
                axes[0, 1].set_xlabel('Recall')
                axes[0, 1].set_ylabel('Precision')
                axes[0, 1].grid(True, alpha=0.3)
                axes[0, 1].set_xlim(0, 1)
                axes[0, 1].set_ylim(0, 1)
            else:
                axes[0, 1].text(0.5, 0.5, 'PR кривая недоступна', ha='center', va='center')
        else:
            axes[0, 1].text(0.5, 0.5, 'PR кривая недоступна', ha='center', va='center')
    except:
        axes[0, 1].text(0.5, 0.5, 'PR кривая недоступна', ha='center', va='center')
    axes[0, 1].set_title('Precision Recall Curve')

    # Confusion Matrix
    try:
        if hasattr(results, 'confusion_matrix') and hasattr(results.confusion_matrix, 'matrix'):
            conf_matrix = results.confusion_matrix.matrix
            if conf_matrix is not None:
                im = axes[0, 2].imshow(conf_matrix, cmap='Blues', aspect='auto')
                axes[0, 2].set_title('Confusion Matrix')
                plt.colorbar(im, ax=axes[0, 2])
                
                class_names = list(model.names.values())
                if len(class_names) <= 10:
                    axes[0, 2].set_xticks(range(len(class_names)))
                    axes[0, 2].set_yticks(range(len(class_names)))
                    axes[0, 2].set_xticklabels(class_names, rotation=45, ha='right')
                    axes[0, 2].set_yticklabels(class_names)
            else:
                axes[0, 2].text(0.5, 0.5, 'Матрица ошибок недоступна', ha='center', va='center')
        else:
            axes[0, 2].text(0.5, 0.5, 'Матрица ошибок недоступна', ha='center', va='center')
    except:
        axes[0, 2].text(0.5, 0.5, 'Матрица ошибок недоступна', ha='center', va='center')
    axes[0, 2].set_title('Confusion Matrix')

    # F1-Confidence кривая
    try:
        if hasattr(results, 'box') and hasattr(results.box, 'f1'):
            f1_curve = results.box.f1
            if f1_curve is not None and len(f1_curve) > 0:
                confidences = np.linspace(0, 1, len(f1_curve))
                axes[1, 0].plot(confidences, f1_curve, 'g-', linewidth=2)
                axes[1, 0].set_title('F1 Confidence Curve')
                axes[1, 0].set_xlabel('Confidence Threshold')
                axes[1, 0].set_ylabel('F1 Score')
                axes[1, 0].grid(True, alpha=0.3)
                
                optimal_idx = np.argmax(f1_curve)
                optimal_conf = confidences[optimal_idx]
                axes[1, 0].axvline(x=optimal_conf, color='r', linestyle='--', alpha=0.7)
                axes[1, 0].text(optimal_conf + 0.05, 0.5, f'Optimal: {optimal_conf:.2f}', color='red')
            else:
                axes[1, 0].text(0.5, 0.5, 'F1 кривая недоступна', ha='center', va='center')
        else:
            axes[1, 0].text(0.5, 0.5, 'F1 кривая недоступна', ha='center', va='center')
    except:
        axes[1, 0].text(0.5, 0.5, 'F1 кривая недоступна', ha='center', va='center')
    axes[1, 0].set_title('F1 Confidence Curve')

    # Распределение confidence scores
    try:
        test_img = 'test.jpg'
        if os.path.exists(test_img):
            test_results = model.predict(test_img, conf=0.25, verbose=False)[0]
            if test_results.boxes is not None and len(test_results.boxes) > 0:
                conf_scores = test_results.boxes.conf.cpu().numpy()
                axes[1, 1].hist(conf_scores, bins=20, alpha=0.7, color='purple', edgecolor='black')
                axes[1, 1].set_title('Распределение Confidence Scores')
                axes[1, 1].set_xlabel('Confidence')
                axes[1, 1].set_ylabel('Частота')
                axes[1, 1].axvline(x=0.5, color='r', linestyle='--', label='Текущий порог 0.5')
                axes[1, 1].legend()
                axes[1, 1].grid(True, alpha=0.3)
            else:
                axes[1, 1].text(0.5, 0.5, 'Нет обнаружений', ha='center', va='center')
        else:
            axes[1, 1].text(0.5, 0.5, 'test.jpg не найден', ha='center', va='center')
    except Exception as e:
        axes[1, 1].text(0.5, 0.5, f'Ошибка: {str(e)[:30]}', ha='center', va='center')
    axes[1, 1].set_title('Распределение Confidence Scores')

    # Метрики по классам
    try:
        class_data = {}
        for i in range(len(model.names)):
            map_key = f'metrics/mAP50(B)/{i}'
            map_key_attr = map_key.replace('/', '_')
            
            if hasattr(results, map_key_attr):
                class_data[model.names[i]] = getattr(results, map_key_attr, 0)
        
        if class_data:
            classes = list(class_data.keys())
            values = list(class_data.values())
            
            y_pos = np.arange(len(classes))
            bars = axes[1, 2].barh(y_pos, values, color=plt.cm.Set3(np.linspace(0, 1, len(classes))))
            axes[1, 2].set_title('mAP@0.5 по классам')
            axes[1, 2].set_xlabel('mAP@0.5')
            axes[1, 2].set_yticks(y_pos)
            axes[1, 2].set_yticklabels(classes)
            axes[1, 2].grid(axis='x', alpha=0.3)
            
            for bar, value in zip(bars, values):
                width = bar.get_width()
                axes[1, 2].text(width + 0.01, bar.get_y() + bar.get_height()/2, 
                              f'{value:.3f}', va='center')
        else:
            axes[1, 2].text(0.5, 0.5, 'Данные по классам недоступны', ha='center', va='center')
    except:
        axes[1, 2].text(0.5, 0.5, 'Данные по классам недоступны', ha='center', va='center')
    axes[1, 2].set_title('Производительность по классам')

    plt.tight_layout()
    plt.savefig('ai-core/model_metrics_visualization.png', dpi=150, bbox_inches='tight')
    plt.close(fig)  # Закрываем фигуру, чтобы не держать в памяти

# ВЫЗОВ ВИЗУАЛИЗАЦИИ
if 'metrics' in locals():
    create_metrics_plots(metrics, "2final.pt")

# ТЕСТИРОВАНИЕ НА ИЗОБРАЖЕНИИ — СОХРАНЯЕМ РЕЗУЛЬТАТЫ В ai-core/
test_image = "test.jpg"
if Path(test_image).exists():
    results = model.predict(test_image, conf=0.5, imgsz=608, verbose=False)
    
    for i, r in enumerate(results):
        output_path = f'ai-core/prediction_{i}.jpg'
        r.save(filename=output_path)
        
        # Также сохраним изображение с наложенной визуализацией
        img = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.title(f'Обнаружение объектов (модель: 2final.pt)')
        plt.axis('off')
        plt.savefig(f'ai-core/detected_{i}.png', dpi=150, bbox_inches='tight')
        plt.close()

# ОЦЕНКА ПРОИЗВОДИТЕЛЬНОСТИ — результаты пишем в файл
if Path(test_image).exists():
    times = []
    for i in range(5):
        start_time = time.time()
        results = model.predict(test_image, conf=0.5, imgsz=608, verbose=False)
        inference_time = time.time() - start_time
        times.append(inference_time)
    
    avg_time = np.mean(times)
    fps = 1 / avg_time if avg_time > 0 else 0
    
    # Сохраняем в файл
    with open('ai-core/performance.txt', 'w') as f:
        f.write(f"Среднее время инференса: {avg_time:.3f} сек\n")
        f.write(f"Примерная скорость: {fps:.1f} FPS\n")

# СОХРАНЕНИЕ РЕЗУЛЬТАТОВ В ФАЙЛ — в ai-core/
try:
    with open('ai-core/evaluation_results.txt', 'w') as f:
        f.write(f"МОДЕЛЬ: {model_path}\n")
        f.write(f"ДАТА ОЦЕНКИ: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("\nМЕТРИКИ КАЧЕСТВА:\n")
        
        f.write(f"mAP@0.5: {map50:.4f}\n")
        f.write(f"mAP@0.5:0.95: {map50_95:.4f}\n")
        f.write(f"Precision: {precision:.4f}\n")
        f.write(f"Recall: {recall:.4f}\n")
        f.write(f"F1 Score: {f1_score:.4f}\n")
        
        f.write("\nСТРУКТУРА ДАННЫХ:\n")
        f.write(f"Конфигурация: {data_yaml_path}\n")
        f.write(f"Классы: {list(model.names.values())}\n")
        
        if Path(test_image).exists():
            test_results = model.predict(test_image, conf=0.5, imgsz=608, verbose=False)[0]
            if test_results.boxes is not None:
                f.write(f"\nТЕСТ НА ИЗОБРАЖЕНИИ:\n")
                f.write(f"Обнаружено объектов: {len(test_results.boxes)}\n")
                conf_scores = test_results.boxes.conf.cpu().numpy()
                f.write(f"Средний confidence: {np.mean(conf_scores):.4f}\n")
except Exception as e:
    with open('ai-core/error.log', 'w') as f:
        f.write(f"Ошибка при сохранении результатов: {e}")


YOLOv10m summary (fused): 136 layers, 15,313,747 parameters, 0 gradients, 58.9 GFLOPs
Ultralytics 8.3.239  Python-3.12.6 torch-2.9.1+cu128 CPU (11th Gen Intel Core(TM) i7-11800H 2.30GHz)
val: Fast image access  (ping: 0.00.0 ms, read: 1943.9600.5 MB/s, size: 696.4 KB)
val: Scanning E:\СИИ\CursachProject\AI-Flow-Detecting\ai-core\dataset\val_split.cache... 84 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 85/85 128.7Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.3s/it 14.0s3.1s
                   all         85        519      0.902      0.318      0.611      0.334
Speed: 0.5ms preprocess, 150.0ms inference, 0.0ms loss, 0.0ms postprocess per image
Saving E:\AIM\AI-Flow-Detecting\runs\detect\val6\predictions.json...
Results saved to E:\AIM\AI-Flow-Detecting\runs\detect\val6
